# Feature Selection in 5 Minutes

## What You Will Do
In this notebook you will apply three fundamentally different feature selection methods to the same dataset and compare what each one finds. By the end you will have a working comparison table and accuracy scores proving that fewer, better-chosen features can match or beat using every feature.

## The Three Methods
1. **Mutual Information Filter** — ranks each feature by how much information it shares with the target, independently of any model
2. **L1 Embedded (Lasso)** — a logistic regression that shrinks unimportant feature coefficients to exactly zero during training
3. **Boruta Wrapper** — compares each feature to a shuffled shadow copy of itself; only keeps features that beat chance across many trials

## Estimated Time: under 2 minutes

---

## Setup

All imports and configuration live here. We pin a random seed so every run produces the same results.

In [ ]:
# Purpose: Import all dependencies and set reproducibility seed
# Key Concept: A fixed random seed means your comparison table will be stable across runs

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

## Load the Dataset

We use the Wisconsin Breast Cancer dataset: 569 tumour samples, 30 numeric features derived from cell nuclei images, binary target (malignant / benign). It is clean, well-understood, and large enough to give meaningful cross-validation estimates.

We deliberately start with **all 30 features** so we can measure the cost of not selecting.

In [ ]:
# Purpose: Load dataset and inspect basic properties
# Key Concept: Understanding the feature space before selection is non-negotiable

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
feature_names = X.columns.tolist()

print(f"Samples : {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes : malignant={int((y == 0).sum())}, benign={int((y == 1).sum())}")
print()
print("First five features:", feature_names[:5])

## Method 1 — Mutual Information Filter

Mutual information (MI) measures how much knowing a feature reduces uncertainty about the target. It captures non-linear relationships that correlation misses.

$$\text{MI}(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)\,p(y)}$$

**Key property**: MI is computed feature-by-feature, ignoring interactions between features. A feature with high individual MI may be redundant if another selected feature carries the same information.

We select the top 10 features by MI score.

In [ ]:
# Purpose: Score every feature with mutual information and select the top 10
# Key Concept: Filter methods score features independently — fast but blind to redundancy

Xn = X.values
yn = y.values

mi_scores = mutual_info_classif(Xn, yn, random_state=RANDOM_STATE)

# Rank features from highest to lowest MI
mi_ranking = np.argsort(mi_scores)[::-1]
TOP_K = 10
mi_selected_idx = mi_ranking[:TOP_K]
mi_selected = [feature_names[i] for i in mi_selected_idx]

print(f"Top {TOP_K} features by mutual information:")
for rank, (idx, name) in enumerate(zip(mi_selected_idx, mi_selected), start=1):
    print(f"  {rank:2d}. {name:35s}  MI = {mi_scores[idx]:.4f}")

## Method 2 — L1 Embedded Selection

Logistic regression with an L1 penalty adds a term to the loss that penalises the sum of absolute coefficient values:

$$\mathcal{L}_{\text{L1}} = -\log\text{-likelihood} + \frac{1}{C}\sum_j |\beta_j|$$

As the penalty strength increases (lower `C`), more coefficients are driven to exactly zero. Features with zero coefficient are discarded. Selection is **embedded** because it happens inside the model fitting process — unlike a filter, the model accounts for feature interactions.

We standardise features first because L1 penalises coefficient magnitude, which is scale-dependent.

In [ ]:
# Purpose: Fit an L1-penalised logistic regression and extract non-zero features
# Key Concept: l1_ratio=1.0 means pure L1 (Lasso); lower C means stronger sparsity

scaler = StandardScaler()
Xs = scaler.fit_transform(Xn)

# l1_ratio=1.0 activates pure L1 sparsity inside the ElasticNet path
l1_model = LogisticRegression(
    C=0.1,
    l1_ratio=1.0,
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE
)
l1_model.fit(Xs, yn)

nonzero_mask = l1_model.coef_[0] != 0
l1_selected = [feature_names[i] for i in np.where(nonzero_mask)[0]]

print(f"Features selected by L1 (non-zero coefficients): {len(l1_selected)} / {len(feature_names)}")
for name in l1_selected:
    coef = l1_model.coef_[0][feature_names.index(name)]
    print(f"  {name:35s}  coef = {coef:+.4f}")

## Method 3 — Boruta Wrapper (from scratch)

Boruta extends a Random Forest by creating **shadow features**: for each real feature, a shuffled (permuted) copy is added to the dataset. A feature earns a "hit" when its Random Forest importance exceeds the maximum importance of any shadow feature. After many trials, features that consistently beat the shadow are declared important.

The elegance: because shadow features carry no information (they are random), beating them repeatedly is a statistically meaningful bar.

This implementation runs 30 trials. A feature is selected if it wins more than 50% of trials.

In [ ]:
# Purpose: Implement Boruta algorithm from scratch using shadow feature comparison
# Key Concept: Wrapper methods evaluate features in the context of a complete model,
#              catching interactions that filters miss

def boruta_select(X_arr, y_arr, n_trials=30, win_threshold=0.5, random_state=42):
    """
    Select features using the Boruta shadow-feature algorithm.

    Parameters
    ----------
    X_arr : ndarray of shape (n_samples, n_features)
    y_arr : ndarray of shape (n_samples,)
    n_trials : int
        Number of shadow-feature trials
    win_threshold : float
        Fraction of trials a feature must win to be selected
    random_state : int

    Returns
    -------
    selected_mask : ndarray of bool, shape (n_features,)
    wins : ndarray of int, shape (n_features,)
    """
    rng = np.random.RandomState(random_state)
    n_features = X_arr.shape[1]
    wins = np.zeros(n_features, dtype=int)

    rf = RandomForestClassifier(
        n_estimators=50,
        max_depth=None,
        random_state=random_state,
        n_jobs=-1
    )

    for trial in range(n_trials):
        # Step 1: create shadow features by permuting each column independently
        X_shadow = np.column_stack([
            rng.permutation(X_arr[:, col]) for col in range(n_features)
        ])

        # Step 2: train RF on real + shadow features concatenated
        X_augmented = np.hstack([X_arr, X_shadow])
        rf.set_params(random_state=rng.randint(0, 100000))
        rf.fit(X_augmented, y_arr)

        # Step 3: compare real importances against shadow maximum
        real_importances = rf.feature_importances_[:n_features]
        shadow_importances = rf.feature_importances_[n_features:]
        shadow_max = shadow_importances.max()

        wins += (real_importances > shadow_max).astype(int)

    selected_mask = wins >= (n_trials * win_threshold)
    return selected_mask, wins


boruta_mask, boruta_wins = boruta_select(Xn, yn, n_trials=30, win_threshold=0.5)
boruta_selected = [feature_names[i] for i in np.where(boruta_mask)[0]]

print(f"Features selected by Boruta: {len(boruta_selected)} / {len(feature_names)}")
for name in boruta_selected:
    w = boruta_wins[feature_names.index(name)]
    print(f"  {name:35s}  wins = {w}/30")

## Comparison Table

Now we put all three methods side by side. Each column shows which features that method selected. The overlap column counts how many methods agreed on a given feature — high agreement is a strong signal.

In [ ]:
# Purpose: Build a comparison table showing method agreement per feature
# Key Concept: Features selected by multiple independent methods are more trustworthy

comparison = pd.DataFrame({
    'Feature': feature_names,
    'MI Filter': ['YES' if f in mi_selected else '   ' for f in feature_names],
    'L1 Embedded': ['YES' if f in l1_selected else '   ' for f in feature_names],
    'Boruta': ['YES' if f in boruta_selected else '   ' for f in feature_names],
})

# Count how many methods selected each feature
comparison['Votes'] = (
    (comparison['MI Filter'] == 'YES').astype(int) +
    (comparison['L1 Embedded'] == 'YES').astype(int) +
    (comparison['Boruta'] == 'YES').astype(int)
)

# Show only features selected by at least one method, sorted by votes
selected_any = comparison[comparison['Votes'] > 0].sort_values('Votes', ascending=False)
print(selected_any.to_string(index=False))

# Consensus set: selected by all three methods
consensus = [f for f in feature_names
             if f in mi_selected and f in l1_selected and f in boruta_selected]
print(f"\nConsensus (all 3 methods): {len(consensus)} features")
print(consensus)

## Accuracy Comparison

A comparison table is only meaningful if we can connect it to predictive performance. We train a Random Forest on five feature sets and compare 5-fold cross-validated accuracy.

The goal is not to pick the best method here — it is to see whether selective subsets pay off against a full-feature baseline.

In [ ]:
# Purpose: Cross-validate a Random Forest on each feature subset
# Key Concept: Cross-validation is essential — a single train/test split can mislead

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_eval = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
)

# Map each selection to its column indices in the original array
def indices_of(names_list):
    return [feature_names.index(n) for n in names_list]

subsets = {
    f'All {len(feature_names)} features': list(range(len(feature_names))),
    f'MI top-10 ({len(mi_selected)})': indices_of(mi_selected),
    f'L1 embedded ({len(l1_selected)})': indices_of(l1_selected),
    f'Boruta ({len(boruta_selected)})': indices_of(boruta_selected),
    f'Consensus ({len(consensus)})': indices_of(consensus),
}

results = {}
for label, idx in subsets.items():
    if len(idx) == 0:
        results[label] = (0.0, 0.0)
        continue
    scores = cross_val_score(rf_eval, Xn[:, idx], yn, cv=cv, scoring='accuracy')
    results[label] = (scores.mean(), scores.std())

print(f"{'Feature Set':<35} {'Mean Acc':>9} {'Std':>7} {'Features':>10}")
print('-' * 65)
for label, (mean, std) in results.items():
    n_feat = subsets[label]
    print(f"{label:<35} {mean:>9.4f} {std:>7.4f} {len(n_feat):>10}")

## Visualisation — Votes and Accuracy

Two charts: the vote histogram shows which features achieved broad agreement, and the accuracy bar chart makes the performance comparison easy to read.

In [ ]:
# Purpose: Visualise feature votes and accuracy comparison
# Key Concept: Visualising method agreement and performance in one view tells the full story

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Feature vote histogram
ax = axes[0]
vote_data = comparison[comparison['Votes'] > 0].sort_values('Votes', ascending=False)
colors_map = {1: '#aec7e8', 2: '#ffbb78', 3: '#98df8a'}
bar_colors = [colors_map[v] for v in vote_data['Votes']]
bars = ax.barh(vote_data['Feature'], vote_data['Votes'], color=bar_colors, edgecolor='white')
ax.set_xlabel('Number of methods selecting this feature', fontsize=11)
ax.set_title('Feature Selection Agreement\n(3 = all methods agree)', fontsize=12)
ax.set_xlim(0, 3.5)
ax.axvline(x=3, color='green', linestyle='--', alpha=0.5, label='Consensus threshold')
legend_patches = [
    mpatches.Patch(color='#aec7e8', label='1 method'),
    mpatches.Patch(color='#ffbb78', label='2 methods'),
    mpatches.Patch(color='#98df8a', label='All 3 methods'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.grid(axis='x', alpha=0.3)

# Right: Accuracy comparison
ax2 = axes[1]
labels = list(results.keys())
means = [results[l][0] for l in labels]
stds = [results[l][1] for l in labels]
x = np.arange(len(labels))
bar_cols = ['#d62728'] + ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']
ax2.bar(x, means, yerr=stds, capsize=5, color=bar_cols[:len(labels)],
        edgecolor='white', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([l.split('(')[0].strip() for l in labels], rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('5-Fold CV Accuracy', fontsize=11)
ax2.set_title('Accuracy: Full Features vs Selected Subsets', fontsize=12)
ax2.set_ylim(0.9, 1.01)
ax2.axhline(y=means[0], color='red', linestyle='--', alpha=0.4, label='Baseline (all features)')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise: Tune the Selection Threshold

**Task:** Modify the code below to experiment with Boruta's `win_threshold` parameter and the MI top-K cutoff. Your goal is to find the smallest feature subset that still beats the full-feature baseline.

**Expected outcome:** You should be able to find a set of fewer than 8 features that achieves accuracy >= 0.96.

<details>
<summary>Hint 1</summary>
Try Boruta with win_threshold=0.7 — requiring 70% wins is a stricter test and produces a smaller, higher-confidence set.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
After increasing the Boruta threshold, check which features are in the new consensus with L1. Even 4-5 features from the worst-perimeter / worst-concave-points group tend to carry most of the discriminative signal.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Experiment with different thresholds

# Change these values
MI_TOP_K = 10          # Try 5, 7, 15
BORUTA_WIN_THRESHOLD = 0.5   # Try 0.6, 0.7, 0.8

# Re-run MI selection with new top-K
mi_selected_exp = [feature_names[i] for i in mi_ranking[:MI_TOP_K]]

# Re-run Boruta with new threshold
boruta_mask_exp, boruta_wins_exp = boruta_select(
    Xn, yn, n_trials=30, win_threshold=BORUTA_WIN_THRESHOLD
)
boruta_selected_exp = [feature_names[i] for i in np.where(boruta_mask_exp)[0]]

# Build experimental consensus
consensus_exp = [f for f in feature_names
                 if f in mi_selected_exp and f in l1_selected and f in boruta_selected_exp]

print(f"MI top-{MI_TOP_K}: {len(mi_selected_exp)} features")
print(f"Boruta (threshold={BORUTA_WIN_THRESHOLD}): {len(boruta_selected_exp)} features")
print(f"Consensus: {len(consensus_exp)} features: {consensus_exp}")

if len(consensus_exp) > 0:
    scores_exp = cross_val_score(
        rf_eval, Xn[:, indices_of(consensus_exp)], yn, cv=cv, scoring='accuracy'
    )
    print(f"\nConsensus accuracy: {scores_exp.mean():.4f} +/- {scores_exp.std():.4f}")
else:
    print("\nNo consensus features — try loosening the thresholds.")

your_best_features = consensus_exp  # Replace with your best selection
your_best_accuracy = scores_exp.mean() if len(consensus_exp) > 0 else 0.0

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise():
    assert your_best_features is not None, "Set your_best_features to your selected feature list."
    assert isinstance(your_best_features, list), "your_best_features should be a list of feature name strings."
    assert len(your_best_features) > 0, "You need at least one feature selected."
    assert your_best_accuracy > 0.93, (
        f"Accuracy {your_best_accuracy:.4f} is below 0.93. "
        "Try adjusting thresholds or using a different subset."
    )
    print(f"Exercise passed! Best accuracy {your_best_accuracy:.4f} with {len(your_best_features)} features.")

test_exercise()

## Summary

**Key Takeaways:**

1. **Filter methods** (mutual information) are fast and model-agnostic but ignore feature interactions.
2. **Embedded methods** (L1) perform selection as part of model training and account for redundancy through coefficient competition.
3. **Wrapper methods** (Boruta) evaluate feature subsets holistically but are more expensive — they run many model fits.
4. **Consensus selection** (features chosen by multiple methods) tends to be conservative and stable.
5. Fewer features often match or exceed full-feature accuracy because noise features hurt generalisation.

**What is next:**
- `time_series_feature_selector.ipynb` — add time structure and avoid data leakage
- `boruta_vs_shap_shootout.ipynb` — deeper comparison with SHAP values
- Module 1 (Statistical Filters) for the mathematical foundations of MI and correlation-based filters